# Fine-tuning Qwen2-VL to say who speaks each line (TRL)

Following the Hugging Face cookbook "Fine-Tuning a Vision Language Model (Qwen2-VL-7B) with the Hugging Face
Ecosystem (TRL)": same model, 4-bit QLoRA, same LoRA settings, `SFTTrainer`. Only the given dataset.

Magiv2 already does the text and reading order well (text_order_score 0.93), so Qwen only does the speakers:
input = the 3 pages + Magiv2's lines numbered in reading order, output = one speaker per line ("7: character2").
Each line's number is also drawn on the page (red box where magi found the text), so Qwen can see which
bubble is which line. Without that (first try) it could not tell where a line is and scored 0.207.
This is only for balanced_joint_f1.

Kaggle: GPU T4 x2 (only one is used), internet on. Inputs: task data and `magi_output.json` (notebook 1).
Changes from the cookbook: T4 has no bf16 so fp16 is used, one GPU only (newer TRL refuses two in a
notebook), LoRA weights kept in float32, pages made smaller so 3 of them fit, batch size 1, the model is
tested in 4-bit, no hub upload / trackio.

In [1]:
!pip install -U -q trl bitsandbytes peft qwen-vl-utils

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 945.8/945.8 kB 26.4 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 47.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 835.4/835.4 kB 37.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.0/35.0 MB 46.6 MB/s eta 0:00:00:00:0100:01


In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"     # use one GPU only (TRL refuses to run when it sees two)

import re
import json
import random
import torch
from PIL import Image, ImageDraw, ImageFont

import sys

DATA = None
MAGI = None
for root, dirs, files in os.walk("/kaggle/input", followlinks=True):
    if "sequences.json" in files:
        DATA = root
    if "magi_output.json" in files:
        MAGI = root + "/magi_output.json"
print(DATA)
print(MAGI)

sys.path.insert(0, DATA)
from score import load_jsonl, tokens, align_tokens

/kaggle/input/datasets/arpitparashar06/manga-task/manga-task-upload/manga-task/dataset
/kaggle/input/datasets/arpitparashar06/manga-task/magi_output.json


## data

In [3]:
system_message = """You are a Vision Language Model that reads manga.
You get 3 consecutive manga pages and the story lines on them, numbered in reading order.
Each line's number is drawn on the page in a red box, where that text is.
For every line say who says it: character1, character2, ... (the same character keeps the same name on all 3 pages)
or NARRATION. Answer with one line per text: "number: speaker"."""

In [4]:
magi = json.load(open(MAGI))
gold = load_jsonl(DATA + "/development/labels.jsonl", reference=True)     # UNKNOWN speakers become None
images = {}
for s in json.load(open(DATA + "/sequences.json")):
    images[s["sequence_id"]] = [DATA + "/" + p for p in s["images"]]

def load_page(path, boxes):
    # draw every line's number in a red box where magi found the text, then make the page
    # smaller so 3 of them fit on a T4 (the text itself comes from magi)
    img = Image.open(path).convert("RGB")
    draw = ImageDraw.Draw(img)
    size = max(24, img.width // 25)
    try:
        font = ImageFont.load_default(size=size)
    except TypeError:
        font = ImageFont.load_default()
    for number, b in boxes:
        draw.rectangle([b[0], b[1], b[2], b[3]], outline=(255, 0, 0), width=5)
        draw.rectangle([b[0], b[1], b[0] + size * 1.3, b[1] + size * 1.1], fill=(255, 0, 0))
        draw.text((b[0] + 3, b[1]), str(number), fill=(255, 255, 255), font=font)
    img.thumbnail((640, 640))
    return img

def magi_lines(sid):
    # magi's story lines in reading order: [page, text, box]
    lines = []
    for p in range(3):
        r = magi[sid]["pages"][p]
        for i in range(len(r["ocr"])):
            if r["keep"][i] and r["ocr"][i].strip() != "":
                lines.append([p, r["ocr"][i].strip(), r["text_boxes"][i]])
    return lines

def gold_speakers(sid, lines):
    # line up magi's text with the gold text (the scorer's own alignment);
    # every magi line gets the gold speaker of most of its words
    out = []
    for p in range(3):
        page = [{"speaker": str(n), "text": lines[n][1]} for n in range(len(lines)) if lines[n][0] == p]
        votes = {}
        for g, pr, w in align_tokens(tokens(gold[sid][p]), tokens(page)):
            if g[1] is not None:
                k = int(pr[1])
                votes.setdefault(k, {})
                votes[k][g[1]] = votes[k].get(g[1], 0) + w
        for x in page:
            k = int(x["speaker"])
            out.append(max(votes[k], key=votes[k].get) if k in votes else None)

    # no gold match (or UNKNOWN) -> previous speaker
    last = None
    for n in range(len(out)):
        if out[n] is None:
            out[n] = last
        last = out[n]
    out = [x if x is not None else "character1" for x in out]

    # character1, character2, ... in order of first appearance
    names = {}
    for n in range(len(out)):
        if out[n] != "NARRATION":
            if out[n] not in names:
                names[out[n]] = "character" + str(len(names) + 1)
            out[n] = names[out[n]]
    return out

def question(lines):
    q = []
    for n in range(len(lines)):
        if n == 0 or lines[n - 1][0] != lines[n][0]:
            q.append("PAGE " + str(lines[n][0] + 1))
        q.append(str(n + 1) + ": " + lines[n][1])
    return "\n".join(q)

sid = list(gold)[0]
lines = magi_lines(sid)
print(question(lines)[:400])
print(gold_speakers(sid, lines)[:10])

PAGE 1
1: Huh? Does komaba seem different than usual?
2: He's doing really well.
3: It looks like he's been working hard for the fall tournament.
4: No, that's not it. It's like he's...
5: out for blood.
PAGE 3
6: Woah!!
7: OKAWA- Senpai, you look so cool!
8: I am not sure if this is a
9: Right here! This is where it happens!
10: So close !!
11: Do you have to keep watching that? It's Embar- rassi
['character1', 'character2', 'character2', 'character1', 'character1', 'character1', 'character3', 'character3', 'character4', 'character4']


In [5]:
def format_data(sid):
    lines = magi_lines(sid)
    pages = []
    for p in range(3):
        boxes = [(n + 1, lines[n][2]) for n in range(len(lines)) if lines[n][0] == p]
        pages.append(load_page(images[sid][p], boxes))
    # one {"type": "image"} per page, the pictures themselves go in "images"
    user_content = []
    for img in pages:
        user_content.append({"type": "image"})
    user_content.append({"type": "text", "text": "Who says each line?\n" + question(lines)})

    messages = [
        {"role": "system", "content": [{"type": "text", "text": system_message}]},
        {"role": "user", "content": user_content},
    ]
    if sid in gold:
        speakers = gold_speakers(sid, lines)
        answer = "\n".join(str(n + 1) + ": " + speakers[n] for n in range(len(lines)))
        messages.append({"role": "assistant", "content": [{"type": "text", "text": answer}]})
    return {"images": pages, "messages": messages}

In [6]:
# same split as my other experiments: 16 dev sequences for validation, 64 for training
ids = list(gold.keys())
random.seed(42)
random.shuffle(ids)
val_ids = ids[:16]
train_ids = ids[16:]
test_ids = [s for s in images if s not in gold]

from datasets import Dataset

train_dataset = Dataset.from_list([format_data(s) for s in train_ids if len(magi_lines(s)) > 0])
eval_dataset = Dataset.from_list([format_data(s) for s in val_ids])
print(len(train_dataset), "train,", len(eval_dataset), "val,", len(test_ids), "test")

63 train, 16 val, 15 test


## model (4-bit, like the cookbook)

In [7]:
from transformers import Qwen2VLForConditionalGeneration, Qwen2VLProcessor, BitsAndBytesConfig

model_id = "Qwen/Qwen2-VL-7B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,      # cookbook uses bfloat16, T4 can't
)

model = Qwen2VLForConditionalGeneration.from_pretrained(
    model_id,
    device_map={"": 0},          # all on GPU 0 (split over two GPUs breaks the loss step)
    torch_dtype=torch.float16,
    quantization_config=bnb_config,
)
processor = Qwen2VLProcessor.from_pretrained(model_id)

config.json:   0%|          | 0.00/1.20k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/56.5k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/730 [00:00<?, ?it/s]

...

preprocessor_config.json:   0%|          | 0.00/347 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

In [8]:
def generate_text_from_sample(model, processor, sample, max_new_tokens=400, device="cuda"):
    text_input = processor.apply_chat_template(sample["messages"][:2], tokenize=False, add_generation_prompt=True)
    model_inputs = processor(text=[text_input], images=sample["images"], return_tensors="pt").to(device)

    generated_ids = model.generate(**model_inputs, max_new_tokens=max_new_tokens)
    trimmed = [out[len(inp):] for inp, out in zip(model_inputs.input_ids, generated_ids)]
    return processor.batch_decode(trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]

# before training
print(generate_text_from_sample(model, processor, eval_dataset[0])[:800])

1: The majority are the result of negative energy that flows out of people...
2: The result of a curse...
3: A curse?
4: It doesn't matter if you believe me or not!
5: Any- WAY...
6: places where crowds gather, like schools or hospitals...
7: ....are susceptible to receiving that energy.
8: Pain, regret, humiliation... the more people dwell on bad memories...
9: Curse
10: Curse
11: ...the more these places become receptive to those feelings.
12: Curse
13: Curse
14: For that reason, a cursed object is often found at such places as a Talisman.
15: The object you found is such an item.
16: A Talisman?
17: Isn't that a good thing? How's that dangerous?
18: A Talisman sounds harmless enough.
19: The more evil placed on a cursed object, the more it defects other causes.
20: Using poison to fend 


## LoRA + training

In [9]:
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

peft_config = LoraConfig(
    lora_alpha=16,
    lora_dropout=0.05,
    r=8,
    bias="none",
    target_modules=["q_proj", "v_proj"],
    task_type="CAUSAL_LM",
)

training_args = SFTConfig(
    output_dir="qwen2-vl-manga",
    num_train_epochs=3,
    per_device_train_batch_size=1,          # cookbook: 4 (3 pages per sample is a lot more memory)
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    max_length=None,
    optim="adamw_torch_fused",
    learning_rate=2e-4,
    logging_steps=5,
    eval_steps=10,
    eval_strategy="steps",
    save_strategy="steps",
    save_steps=20,
    fp16=True,                               # cookbook: bf16=True
    max_grad_norm=0.3,
    warmup_steps=1,                          # cookbook: warmup_ratio=0.03 (removed in new transformers)
    push_to_hub=False,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    peft_config=peft_config,
    processing_class=processor,
)

# fp16 training needs the trainable (LoRA) weights in float32, the rest stays 4-bit
for p in trainer.model.parameters():
    if p.requires_grad:
        p.data = p.data.float()

In [10]:
trainer.train()
trainer.save_model(training_args.output_dir)

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 151645, 'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
10,12.406498,12.534121,4.657499,0.187424,127667.000000
20,11.220340,11.433125,5.802852,0.187630,255979.000000
24,11.220340,11.287235,5.929179,0.187627,306078.000000


## test the fine-tuned model

In [11]:
import gc
import time

def clear_memory():
    for name in ["inputs", "model", "processor", "trainer"]:
        if name in globals():
            del globals()[name]
    time.sleep(2)
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    print(f"GPU allocated memory: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

clear_memory()

# 4-bit again: the full-size model does not fit on one T4
model = Qwen2VLForConditionalGeneration.from_pretrained(
    model_id, device_map={"": 0}, torch_dtype=torch.float16, quantization_config=bnb_config)
processor = Qwen2VLProcessor.from_pretrained(model_id)
model.load_adapter("qwen2-vl-manga")

GPU allocated memory: 1.03 GB


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/730 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/112 [00:00<?, ?it/s]

LoadStateDictInfo(missing_keys=set(), unexpected_keys=set(), mismatched_keys=set(), error_msgs=[], conversion_errors={}, skipped_pp_keys=set())

In [12]:
# model answer -> the task's json format (text from magi, speakers from qwen)
def parse(text, lines):
    said = {}
    for row in text.split("\n"):
        m = re.match(r"\s*(\d+)\s*:\s*(\S+)", row)
        if m:
            said[int(m.group(1)) - 1] = m.group(2)
    pages = [[], [], []]
    last = "character1"
    for n in range(len(lines)):
        spk = said.get(n, last)
        # anything that is not characterN / NARRATION (e.g. UNKNOWN) -> previous speaker
        if spk != "NARRATION" and not re.fullmatch(r"character\d+", spk):
            spk = last
        pages[lines[n][0]].append({"speaker": spk, "text": lines[n][1]})
        last = spk
    return pages

out = generate_text_from_sample(model, processor, eval_dataset[0])
print(out[:500])
print(parse(out, magi_lines(val_ids[0]))[0][:5])

number: The majority are the result of negative energy that flows out of people...
number: A curse?
number: Any- WAY...
number: Pain, regret, humiliation... the more people dwell on bad memories...
number: Curse
number: Curse
number: Curse
number: Curse
number: Curse
number: Curse
number: Curse
number: Curse
number: Curse
number: Curse
number: Curse
number: Curse
number: Curse
number: Curse
number: Curse
number: Curse
number: Curse
number: Curse
number: Curse
number: Curse
number: Curse
number: 
[{'speaker': 'character1', 'text': 'The majority are the result of negative energy that flows out of people...'}, {'speaker': 'character1', 'text': 'The result of a curse...'}, {'speaker': 'character1', 'text': 'A curse?'}, {'speaker': 'character1', 'text': "It doesn't matter if you believe me or not!"}, {'speaker': 'character1', 'text': 'Any- WAY...'}]


In [14]:
# all 16 validation sequences
f = open("/kaggle/working/val_labels.jsonl", "w")
for line in open(DATA + "/development/labels.jsonl"):
    if json.loads(line)["sequence_id"] in val_ids:
        f.write(line)
f.close()

f = open("/kaggle/working/qwen_val.jsonl", "w")
for i, sid in enumerate(val_ids):
    out = generate_text_from_sample(model, processor, eval_dataset[i])
    f.write(json.dumps({"sequence_id": sid, "pages": parse(out, magi_lines(sid))}) + "\n")
    print(i, sid)
f.close()

!python {DATA}/score.py --references /kaggle/working/val_labels.jsonl --predictions /kaggle/working/qwen_val.jsonl --output /kaggle/working/qwen_val_scores.json


0 seq_ec05f45d71c43f2c
1 seq_07838fb26cd1be54
2 seq_96839890d69d7698
3 seq_de542b63e6b19bc2
4 seq_d8bc3c6ac1bb6017
5 seq_84541a1f4b159018
6 seq_ee880b9612205860
7 seq_cd727990f22c777b
8 seq_4833e23eec54344b
9 seq_eaad741293da2ac1
10 seq_d8afceb0e8f9122d
11 seq_58858a2227cf3761
12 seq_c4448514b6de1ab9
13 seq_0ef2ec294597f572
14 seq_fc4c2b84b6ec32ff
15 seq_5f407d0064bb4a43
{
  "text_order_score": 0.9407950002777625,
  "balanced_joint_f1": 0.2635085908331202,
  "speaker_accuracy_on_matched": 0.5996565239270061,
  "joint_f1": 0.5697924590390938,
  "matched_token_coverage": 0.9575425989366222
}


In [ ]:
# the 15 test sequences
f = open("/kaggle/working/qwen_test_predictions.jsonl", "w")
for sid in test_ids:
    out = generate_text_from_sample(model, processor, format_data(sid))
    f.write(json.dumps({"sequence_id": sid, "pages": parse(out, magi_lines(sid))}) + "\n")
    print(sid)
f.close()

seq_6b18300406fa878f
seq_c780ac6d1dbd1242
seq_5303b549f62e9d3f
seq_d455cfd18edfb0ba
seq_e04ef3d20fd0baa8
seq_6aa313a0c0fa1c39
seq_32c47ca11b40ca8a
seq_81b1782990ab7950
